In [1]:
import json
import os, sys
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
import seaborn as sns
import ROOT as rt


## user define
sys.path.append(".")
sys.path.append("./Utils")
from Plotter import HistStack
from CommonTrain import *
from Model import Seq

2023-05-28 14:37:18.870048: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorfl

Welcome to JupyROOT 6.26/08


# Set config

In [2]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
INPUTDIR = setting["INPUTDIR"]
VERSION = setting["VERSION"]
# SIGNAL_MASSES = ["Zp-1700_CH-345", "Zp-2900_CH-345", "Zp-4100_CH-345", "Zp-2900_CH-1095", "Zp-4100_CH-1095", "Zp-4100_CH-1845"]

# ERA="ee17"
print(VERSION)
INPUTDIR = sys.argv[1]
REGION = sys.argv[2]
PART = sys.argv[3]

# # for dev
# VERSION = '20230527deepCSV-SR2'
# CHANNEL="ee"
# # INPUTDIR = setting["INPUTDIR"]+'/em18/'
# INPUTDIR = '/eos/user/h/hkwon/Run2/Preselection/20230526deepCSV/Presel/ee16/'
# # INPUTDIR = '/eos/user/h/hkwon/SWAN_projects/Zpeak/20230502pdf_17/Presel/ee17/'
# REGION = 'SR'
# PART = 'dummy'

20230527deepCSV


# Load data

In [3]:
if REGION == 'SR':
    file = 'nested_dic_df_SR1.pkl'
if REGION == 'VR':   
    file = 'nested_dic_df_TTCR.pkl'
#     file = 'nested_dic_df_PT.pkl'
# file = "/"+ERA+"/nested_dic_df.pkl"
with open(INPUTDIR+'/'+file, 'rb') as f:
    nested_dic_df = pickle.load(f)
 
dic_df=nested_dic_df['raw']
dic_df_met_u_unsort=nested_dic_df['met_u']
dic_df_met_d_unsort=nested_dic_df['met_d'] 
dic_df_met_ue_u_unsort=nested_dic_df['met_ue_u']
dic_df_met_ue_d_unsort=nested_dic_df['met_ue_d'] 

# # temp!!
for key in dic_df_met_u_unsort:
    df = dic_df_met_u_unsort[key]
#     print(df[df.isin([np.inf, -np.inf]).any(1)])
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_u_unsort[key] = df.drop(i)
    df = dic_df_met_d_unsort[key]
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_d_unsort[key] = df.drop(i)
        
selected_columns = [
#                     'MC1C1_pre', 'H3PP_pre', 'H3TPP_pre', 
                    'pt_lead_pre', 'mass_pre', # by auc rank
                    'MT_pre', 'pt_trail_pre', 'Zpt_pre', 'dRll_pre', 'PuppiMET_pre',
                    'dPhilMET_pre', 'dPhillMET_pre', 'U_pre', 'dPhill_pre', 'MT2_pre',
                    'JZB_pre'] # order used in training

# # dic_df_meta={}
# # check b jet
# query_str="nbjet_pre==0"

# print("####################################################")
# print("####   (unweighted) #events after preselection  ####")
# print("####################################################")
# for key in dic_df:  
# #     dic_df_meta[key]=dic_df[key].query(query_str)
#     dic_df[key]=dic_df[key].query(query_str) #temp!
#     print(key, ":", dic_df[key].shape[0])
    
dic_df_train={}    
dic_df_met_u={}
dic_df_met_d={}
dic_df_met_ue_u={}
dic_df_met_ue_d={}
for key in dic_df:
    print(key)
    #dic_df_train[key]=pd.concat([dic_df[key][selected_columns], dic_df1[key][selected_columns]]) #when use ll channel
    #dic_df[key]=pd.concat([dic_df[key], dic_df1[key]])
#     dic_df_train[key]=dic_df_meta[key][selected_columns] # check b jet   
    dic_df_train[key]=dic_df[key][selected_columns]    
    dic_df_met_u[key]=dic_df_met_u_unsort[key][selected_columns]    
    dic_df_met_d[key]=dic_df_met_d_unsort[key][selected_columns]    
    dic_df_met_ue_u[key]=dic_df_met_ue_u_unsort[key][selected_columns]    
    dic_df_met_ue_d[key]=dic_df_met_ue_d_unsort[key][selected_columns]

Zp-4100_CH-1345
DY1500To2000
Zp-1700_CH-345
Zp-2500_CH-345
WWW
DY500To700
TTSemi
Zp-1700_CH-845
Zp-4100_CH-595
Zp-3300_CH-345
Zp-1700_CH-595
Zp-3700_CH-345
Zp-2100_CH-595
Zp-2900_CH-345
DY
WZQQ
Zp-4100_CH-1095
Zp-2900_CH-845
Zp-3700_CH-1345
WW
Zp-3300_CH-1595
Zp-2500_CH-595
Zp-4100_CH-345
Zp-3700_CH-1845
TT
Zp-2500_CH-845
WZ
ZZ
Zp-4100_CH-1595
Zp-4100_CH-1845
Zp-3300_CH-595
H
Zp-2100_CH-345
Zp-2900_CH-595
top
Zp-4100_CH-845
Zp-2100_CH-845
Zp-3300_CH-845
antitop
Zp-3300_CH-1345
Zp-3700_CH-1595
DY2000To3000
HWW
Zp-3700_CH-595
TThdampUp
TTDown
TTUp
DY700To800
Zp-2900_CH-1345
WWZ
Zp-3700_CH-1095
DY800To1000
DY1000To1500
Zp-3700_CH-845
TTZ
Zp-3300_CH-1095
TThdampDown
Zp-2900_CH-1095
Data


In [4]:
print(dic_df["Zp-2500_CH-345"].columns)
print(dic_df_train["Zp-2500_CH-345"].columns)
print(dic_df_met_u["Zp-2500_CH-345"].columns)

Index(['PuppiMET_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre',
       ...
       'weighteffSFUp', 'weighteffSFDown', 'weighttopSFUp', 'weighttopSFDown',
       'weightgenweightUp', 'weightgenweightDown', 'weighttrigSFUp',
       'weighttrigSFDown', 'weightbtagSFUp', 'weightbtagSFDown'],
      dtype='object', length=169)
Index(['pt_lead_pre', 'mass_pre', 'MT_pre', 'pt_trail_pre', 'Zpt_pre',
       'dRll_pre', 'PuppiMET_pre', 'dPhilMET_pre', 'dPhillMET_pre', 'U_pre',
       'dPhill_pre', 'MT2_pre', 'JZB_pre'],
      dtype='object')
Index(['pt_lead_pre', 'mass_pre', 'MT_pre', 'pt_trail_pre', 'Zpt_pre',
       'dRll_pre', 'PuppiMET_pre', 'dPhilMET_pre', 'dPhillMET_pre', 'U_pre',
       'dPhill_pre', 'MT2_pre', 'JZB_pre'],
      dtype='object')


In [5]:
def get_x(sig_key, dic_df_train):
    # x: sig and all bkgs concatenated
    x = dic_df_train[sig_key].copy(deep=True)
    for k in dic_df_train:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_df_train[k]], axis=0)
    
    return x

In [6]:
def get_xy(sig_key, dic_predict_sort):
    # x: sig and all bkgs concatenated
    x = dic_predict_sort[sig_key].copy(deep=True)
    for k in dic_predict_sort:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_predict_sort[k]], axis=0)
            
    tot_sig = dic_predict_sort[sig_key].shape[0]
    tot_bkgs = x.shape[0] - tot_sig        
    y_sig = np.full((tot_sig,1), 1)
    y_bkgs = np.full((tot_bkgs,1), 0) 
    y = np.concatenate((y_sig, y_bkgs), axis=0) 
    
    return x, y

In [7]:
import math
SYS_LIST = ["puweight", "l1prefiring", "effSF", "topSF", "met", "metue", "trigSF", "hdamp", "PS", "btagSF"]

# sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "met", "trigSF"]
sys_list = SYS_LIST

# sys_list = ["l1prefiring"]
if REGION == 'SR':
    threshold = 0.6
else:
    threshold = 1    
    
def draw_with_systematics(model_mass, dic_df_raw, dic_nested_shape_sys, sys_list, variable, r, region):
    dic_sys = {}
    dic_sys_sqr = {}
    # nominal weight plot
    file_prefix = "Outputs/"+REGION
    hist=HistStack(dic_df_raw, variable, 'weight', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/norm_dnnscore"+VERSION+".pdf")     
    dic_yields = hist.get_yields_by_bin()
    bkg_norm = dic_yields['bkg']
    hist.create_root_file(file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    
    for s in sys_list:
        if "met" in s or "scale" in s: # fill later
            continue
        # up
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Up', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systUp_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysUp = dic_yields['bkg']
        dic_sys[s+'Up'] = np.array(sysUp) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        # down
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Down', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systDown_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysDown = dic_yields['bkg']
        dic_sys[s+'Down'] = np.array(sysDown) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        
    # shape sys
    hist=HistStack(dic_nested_shape_sys['met_u'], variable, 'weightmetUp', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systUp_met.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    sysUp = dic_yields['bkg']
    dic_sys['metUp'] = np.array(sysUp) - np.array(bkg_norm)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    hist=HistStack(dic_nested_shape_sys['met_d'], variable, 'weightmetDown', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systDown_met.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    sysUp = dic_yields['bkg']
    dic_sys['metDown'] = np.array(sysUp) - np.array(bkg_norm)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)

    hist=HistStack(dic_nested_shape_sys['met_ue_u'], variable, 'weightmetueUp', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systUp_metue.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    sysUp = dic_yields['bkg']
    dic_sys['metueUp'] = np.array(sysUp) - np.array(bkg_norm)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    hist=HistStack(dic_nested_shape_sys['met_ue_d'], variable, 'weightmetueDown', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systDown_metue.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    sysUp = dic_yields['bkg']
    dic_sys['metueDown'] = np.array(sysUp) - np.array(bkg_norm)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    
    # si^2
    for s in sys_list:
        # by source
        sqrUp = [x*x if x>0 else -x*x for x in dic_sys[s+'Up']]
        sqrUp_up = [x if x>0 else 0 for x in sqrUp]
        sqrUp_down = [x if x<0 else 0 for x in sqrUp]
        sqrDown = [x*x if x>0 else -x*x for x in dic_sys[s+'Down']]
        sqrDown_up = [x if x>0 else 0 for x in sqrDown]
        sqrDown_down = [x if x<0 else 0 for x in sqrDown]
        
        sqr_up = np.array(sqrUp_up) + np.array(sqrDown_up)
        sqr_down = np.array(sqrUp_down) + np.array(sqrDown_down)
        
        dic_sys_sqr[s+'Up'] = sqr_up
        dic_sys_sqr[s+'Down'] = np.fabs(sqr_down) #store absolute
        
    print(dic_sys)
    print(dic_sys_sqr)
    # sqrt(sumi si^2)
    for i, s in enumerate(sys_list):
        if i==0:
            sum_up = dic_sys_sqr[s+'Up'].copy() # be carefull not to use shallow copy
            sum_down = dic_sys_sqr[s+'Down'].copy()
        else:
            sum_up += dic_sys_sqr[s+'Up']
            sum_down += dic_sys_sqr[s+'Down']

    dic_sys_sqr['sum_up'] = np.sqrt(sum_up)
#     print(sum_up)
#     print(sum_down)
    dic_sys_sqr['sum_down'] = np.sqrt(sum_down)
    print(dic_sys_sqr)
            
    return dic_sys_sqr

In [8]:
from multiprocessing import Pool
def wrapper_mp(model_mass):
    
#     # dump roc
#     dic_roc = {}
#     x, y = get_xy(model_mass, dic_predict_sort)
#     tpr, fpr, auc = dump_roc(model_mass, y, x, dic_predict_sort)
#     dic_roc['tpr'] = tpr
#     dic_roc['fpr'] = fpr
#     dic_roc['auc'] = auc
#     dic_nested_roc[model_mass] = dic_roc            


    #-- implement systematics --#   
    os.system("mkdir -p Outputs/"+REGION)
    os.system("mkdir -p Outputs/"+REGION+"/sys")
#     sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "trigSF"]
    # sys_list = ["puweight"]

    r = [nbin, 0, 1, "DNN score"]

    if REGION == 'SR':
        threshold = 0.6
    else:
        threshold = 1    
    
    # draw plot with syst
    dic_sys_sqr = draw_with_systematics(model_mass, dic_predict_sort, dic_nested_shape_sys, sys_list, "score"+model_mass, r, "region_dummy")

    hist=HistStack(dic_predict_sort, "score"+model_mass, 'weight', nbin, 0, 1, "DNN score", dic_sys=dic_sys_sqr) 
    
    c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/"+model_mass+VERSION, blind_from=threshold)     
    hist.get_yeilds()
    hist.draw_pie(file_name="Outputs/"+REGION+"/"+model_mass+VERSION)     
    hist.export_yields_table(file_name="Outputs/"+REGION+"/"+model_mass)     
     
    return None

In [9]:
dic_df_train['TT'].keys()

Index(['pt_lead_pre', 'mass_pre', 'MT_pre', 'pt_trail_pre', 'Zpt_pre',
       'dRll_pre', 'PuppiMET_pre', 'dPhilMET_pre', 'dPhillMET_pre', 'U_pre',
       'dPhill_pre', 'MT2_pre', 'JZB_pre'],
      dtype='object')

# Load model and evaluate

In [10]:
import time

nbin = 20
if REGION == "VR":
    nbin = 10
# VERSION = '0903'
# VERSION = setting["VERSION"]
print(time.time())

one_mass = 'Zp-2500_CH-345'

if 'mm' in INPUTDIR:
    CHANNEL = 'mm'
if 'ee' in INPUTDIR:
    CHANNEL = 'ee'
if 'em' in INPUTDIR:
    CHANNEL = 'em'
    
OUTDIR = 'Outputs/'+CHANNEL
file_postfix = 'PNN_'+VERSION
# file_postfix = one_mass+VERSION

# model_masses = ['Zp-1700_CH-345', 'Zp-2500_CH-345', 'Zp-2900_CH-345', 'Zp-3300_CH-345', 'Zp-4100_CH-345']
model_masses = [m for m in dic_df_train if 'Zp' in m] # if want to run all
if PART=="one":
    model_masses = [m for m in dic_df_train if 'Zp-3' in m or 'Zp-4' in m]
if PART=="two":
    model_masses = [m for m in dic_df_train if 'Zp-1' in m or 'Zp-2' in m]
# model_masses = ['Zp-2500_CH-345', 'Zp-4100_CH-345']

# %matplotlib inline
rt.gROOT.SetBatch(rt.kTRUE)

dic_nested_roc = {}

ERA = INPUTDIR

model = keras.models.load_model(OUTDIR+'/model_'+file_postfix)
#     model = keras.models.load_model(OUTDIR+'/model_binary_'+file_postfix)

Transformer = pickle.load(open(OUTDIR+'/scaler_'+file_postfix+'.pkl','rb'))

nested_dic_predict, nested_dic_predict_met_u, nested_dic_predict_met_d, nested_dic_predict_met_ue_u, nested_dic_predict_met_ue_d = [{} for _ in range(5)]

# masses loop
for model_mass in model_masses:
    
    masses = model_mass.split('_')
    zp_m = masses[0].split('-')
    zp_m = zp_m[1]
    ch_m = masses[1].split('-')
    ch_m = ch_m[1]

#     file_postfix = 'test'

    dic_df_train_tmp, dic_df_met_u_tmp, dic_df_met_d_tmp, dic_df_met_ue_u_tmp, dic_df_met_ue_d_tmp = [{} for _ in range(5)]        
    dic_df_scale, dic_df_scale_met_u, dic_df_scale_met_d, dic_df_scale_met_ue_u, dic_df_scale_met_ue_d = [{} for _ in range(5)]        
    
    
    for key2 in dic_df_train:
        dic_df_train_tmp[key2] = dic_df_train[key2].copy(deep=True)
        dic_df_met_u_tmp[key2] = dic_df_met_u[key2].copy(deep=True)
        dic_df_met_d_tmp[key2] = dic_df_met_d[key2].copy(deep=True)
        dic_df_met_ue_u_tmp[key2] = dic_df_met_ue_u[key2].copy(deep=True)
        dic_df_met_ue_d_tmp[key2] = dic_df_met_ue_d[key2].copy(deep=True)
        dic_df_train_tmp[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
        dic_df_train_tmp[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))       
        dic_df_met_u_tmp[key2]['zp_m'] = np.full(dic_df_met_u[key2].shape[0], int(zp_m))
        dic_df_met_u_tmp[key2]['ch_m'] = np.full(dic_df_met_u[key2].shape[0], int(ch_m))
        dic_df_met_d_tmp[key2]['zp_m'] = np.full(dic_df_met_d[key2].shape[0], int(zp_m))
        dic_df_met_d_tmp[key2]['ch_m'] = np.full(dic_df_met_d[key2].shape[0], int(ch_m))        
        dic_df_met_ue_u_tmp[key2]['zp_m'] = np.full(dic_df_met_ue_u[key2].shape[0], int(zp_m))
        dic_df_met_ue_u_tmp[key2]['ch_m'] = np.full(dic_df_met_ue_u[key2].shape[0], int(ch_m))          
        dic_df_met_ue_d_tmp[key2]['zp_m'] = np.full(dic_df_met_ue_d[key2].shape[0], int(zp_m))
        dic_df_met_ue_d_tmp[key2]['ch_m'] = np.full(dic_df_met_ue_d[key2].shape[0], int(ch_m))       

        if dic_df_train[key2].empty:
            continue
        dic_df_scale[key2] = Transformer.transform(dic_df_train_tmp[key2]) 
        dic_df_scale_met_u[key2] = Transformer.transform(dic_df_met_u_tmp[key2]) 
        dic_df_scale_met_d[key2] = Transformer.transform(dic_df_met_d_tmp[key2])         
        dic_df_scale_met_ue_u[key2] = Transformer.transform(dic_df_met_ue_u_tmp[key2]) 
        dic_df_scale_met_ue_d[key2] = Transformer.transform(dic_df_met_ue_d_tmp[key2]) 

    # predict
    dic_predict, dic_predict_met_u, dic_predict_met_d, dic_predict_met_ue_u, dic_predict_met_ue_d = [{} for _ in range(5)]

    for key3 in dic_df_scale:
    #             if "Zp-" in key3 and not key in key3: # use same scaling used in model training for all masses
    #                 continue
        dic_predict[key3] = model.predict(dic_df_scale[key3], batch_size=dic_df_scale[key3].shape[0])
        dic_predict_met_u[key3] = model.predict(dic_df_scale_met_u[key3], batch_size=dic_df_scale_met_u[key3].shape[0])
        dic_predict_met_d[key3] = model.predict(dic_df_scale_met_d[key3], batch_size=dic_df_scale_met_d[key3].shape[0])        
        dic_predict_met_ue_u[key3] = model.predict(dic_df_scale_met_ue_u[key3], batch_size=dic_df_scale_met_ue_u[key3].shape[0])
        dic_predict_met_ue_d[key3] = model.predict(dic_df_scale_met_ue_d[key3], batch_size=dic_df_scale_met_ue_d[key3].shape[0])
    
    nested_dic_predict[model_mass] = dic_predict
    nested_dic_predict_met_u[model_mass] = dic_predict_met_u
    nested_dic_predict_met_d[model_mass] = dic_predict_met_d    
    nested_dic_predict_met_ue_u[model_mass] = dic_predict_met_ue_u
    nested_dic_predict_met_ue_d[model_mass] = dic_predict_met_ue_d
    

# array in y-axis to dataframe
dic_predict_sort, dic_predict_met_u_sort, dic_predict_met_d_sort, dic_predict_met_ue_u_sort, dic_predict_met_ue_d_sort = [{} for _ in range(5)]

for key4 in dic_df_scale:
    new_df = dic_df[key4].copy(deep=True).reset_index(drop="True")
    new_df_met_u = pd.DataFrame()
    new_df_met_d = pd.DataFrame()    
    new_df_met_ue_u = pd.DataFrame()
    new_df_met_ue_d = pd.DataFrame()
    new_df_met_u["weightmetUp"] = dic_df_met_u_unsort[key4]["weightmetUp"].copy(deep=True).reset_index(drop="True")
    new_df_met_d["weightmetDown"] = dic_df_met_d_unsort[key4]["weightmetDown"].copy(deep=True).reset_index(drop="True")        
    new_df_met_ue_u["weightmetueUp"] = dic_df_met_ue_u_unsort[key4]["weightmetueUp"].copy(deep=True).reset_index(drop="True")
    new_df_met_ue_d["weightmetueDown"] = dic_df_met_ue_d_unsort[key4]["weightmetueDown"].copy(deep=True).reset_index(drop="True")    
    dic_predict_sort[key4] = new_df
    dic_predict_met_u_sort[key4] = new_df_met_u
    dic_predict_met_d_sort[key4] = new_df_met_d    
    dic_predict_met_ue_u_sort[key4] = new_df_met_ue_u
    dic_predict_met_ue_d_sort[key4] = new_df_met_ue_d
    
for model_mass in model_masses:
    for key5 in dic_df_scale:
        dic_predict_sort[key5]["score"+model_mass] = nested_dic_predict[model_mass][key5][:,0]
        dic_predict_met_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_u[model_mass][key5][:,0]
        dic_predict_met_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_d[model_mass][key5][:,0]        
        dic_predict_met_ue_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_ue_u[model_mass][key5][:,0]
        dic_predict_met_ue_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_ue_d[model_mass][key5][:,0]
    
dic_nested_shape_sys={}
dic_nested_shape_sys["met_u"]=dic_predict_met_u_sort
dic_nested_shape_sys["met_d"]=dic_predict_met_d_sort  
dic_nested_shape_sys["met_ue_u"]=dic_predict_met_ue_u_sort
dic_nested_shape_sys["met_ue_d"]=dic_predict_met_ue_d_sort         
with Pool() as p:
    p.map(wrapper_mp, model_masses)  


with open(OUTDIR+'/nested_dic_roc_'+file_postfix+'.pkl', 'wb') as f:
    pickle.dump(dic_nested_roc, f)            
        


1685277461.9403336
{'puweightUp': array([-4.08165213,  3.92656255,  6.38255219,  1.21534658,  0.12644595,
        0.16535899, -0.58730228, -0.37629344,  0.65469455, -0.28636829,
        1.66752495, -0.05231656, -0.71746351, -0.17220594,  0.19936023,
       -0.21366913, -0.22443622,  0.03385026, -0.17683311,  0.09890392]), 'puweightDown': array([-0.11297881, -1.53728182, -2.94674579, -1.10345711, -0.5092089 ,
        0.08662449,  0.58507126,  0.1104542 , -0.3977393 ,  0.56219409,
       -1.10603052, -0.1311814 ,  0.69033126,  0.72850718, -0.17619317,
       -0.07577838,  0.33683642, -0.11356895,  0.27091332,  0.10740377]), 'l1prefiringUp': array([ 8.34935004e+00,  5.93703683e+00,  2.38014426e+00,  7.91707546e-01,
        5.88937365e-01,  5.77478032e-01,  4.70914398e-01,  2.14452550e-01,
        2.12009903e-01, -5.24455219e-03,  2.99642979e-01,  1.45669068e-01,
        1.37382980e-01,  1.54983622e-01,  2.32546292e-01,  1.95341551e-01,
        2.08211247e-01,  2.99623319e-01,  1.24745517e

       4.91216246e-01, 2.63717367e-01, 9.03172612e-02, 3.85628134e-01])}
{'puweightUp': array([0.00000000e+00, 1.54178934e+01, 4.07369725e+01, 1.47706732e+00,
       1.59885774e-02, 3.48473979e-02, 3.42308380e-01, 1.22001313e-02,
       4.28624960e-01, 3.16062192e-01, 2.78063945e+00, 0.00000000e+00,
       4.76557242e-01, 5.30722707e-01, 3.97444995e-02, 0.00000000e+00,
       1.13458773e-01, 1.14583998e-03, 7.33940294e-02, 2.13175546e-02]), 'puweightDown': array([1.66726483e+01, 2.36323539e+00, 8.68331074e+00, 1.21761759e+00,
       2.59293707e-01, 0.00000000e+00, 3.44923966e-01, 1.41596750e-01,
       1.58196554e-01, 8.20067950e-02, 1.22330351e+00, 1.99455827e-02,
       5.14753882e-01, 2.96548872e-02, 3.10440333e-02, 5.13968599e-02,
       5.03716159e-02, 1.28979063e-02, 3.12699491e-02, 0.00000000e+00]), 'l1prefiringUp': array([6.97116461e+01, 3.52484063e+01, 5.66508671e+00, 6.26800839e-01,
       3.46847220e-01, 3.33480877e-01, 2.21760370e-01, 4.59898962e-02,
       4.49481990e-02, 

array('f', [5.518579483032227, 2.3303170204162598, 11.783997535705566, 61.84458923339844, 344.1114807128906, 224.3502197265625, 1451.99072265625, 1464.316162109375, 1439.502685546875, 1458.5693359375, 1446.3861083984375, 164.14505004882812, 12.901274681091309])
test
{'puweightUp': array([-1.43570228,  9.1162436 ,  1.60935234, -0.25626549,  0.13741759,
        0.15747749, -0.55600925, -0.04108401, -0.16681553, -0.15904296,
       -0.08215138, -0.43276691, -0.28225319,  0.26672308, -0.26082055,
       -0.07986882,  0.01908396, -0.27363284,  0.2792346 ,  0.02294012]), 'puweightDown': array([-0.54118454, -5.57499687, -0.82720749,  0.42770531, -0.03059988,
        0.0649315 ,  0.48092953, -0.21690746,  0.02189374,  0.11862727,
       -0.03005207,  0.51110425,  0.2483259 ,  0.16638145,  0.3977899 ,
       -0.1717552 , -0.02368585,  0.46807229, -0.17039187, -0.05080809]), 'l1prefiringUp': array([13.57036544,  4.73617241,  0.83684168,  0.35723804,  0.37472464,
        0.1538512 ,  0.13081833, 

       6.21204639e-02, 3.61655915e-01, 2.23684146e-02, 5.46540242e-03])}
{'puweightUp': array([0.00000000e+00, 8.31058974e+01, 2.59001495e+00, 1.82931835e-01,
       1.88835929e-02, 2.90152582e-02, 2.31293211e-01, 0.00000000e+00,
       4.79335696e-04, 1.40724300e-02, 0.00000000e+00, 2.61227554e-01,
       6.16657541e-02, 9.88239914e-02, 1.58236801e-01, 0.00000000e+00,
       3.64197438e-04, 2.19091670e-01, 7.79719618e-02, 5.26249104e-04]), 'puweightDown': array([2.35412174e+00, 3.10805901e+01, 6.84272235e-01, 6.56720019e-02,
       9.36352528e-04, 0.00000000e+00, 3.09146287e-01, 4.87367401e-02,
       2.78274203e-02, 2.52946628e-02, 7.65197570e-03, 1.87287197e-01,
       7.96668622e-02, 0.00000000e+00, 6.80273611e-02, 3.58788758e-02,
       5.61019391e-04, 7.48749321e-02, 2.90333901e-02, 2.58146172e-03]), 'l1prefiringUp': array([1.84154818e+02, 2.24313291e+01, 7.00303993e-01, 1.27619014e-01,
       1.40418555e-01, 2.36701933e-02, 1.71134367e-02, 2.76281440e-02,
       1.30912696e-02, 

array('f', [5.518579483032227, 2.3303170204162598, 11.783997535705566, 61.84458923339844, 344.1114807128906, 224.3502197265625, 1451.99072265625, 1464.316162109375, 1439.502685546875, 1458.5693359375, 1446.3861083984375, 164.14505004882812, 12.901274681091309])
test


2023-05-28 14:37:42.446034: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow/contrib

In [11]:
dic_df_train['H']

,pt_lead_pre,mass_pre,MT_pre,pt_trail_pre,Zpt_pre,dRll_pre,PuppiMET_pre,dPhilMET_pre,dPhillMET_pre,U_pre,dPhill_pre,MT2_pre,JZB_pre
164,231.189178,102.357971,58.866024,172.547165,403.480133,0.507252,133.037399,0.08374,0.052962,536.377197,0.072021,1.77934,132.897079


# signal unc.

In [12]:
def write_sys_hist(h1, h2):
    file_name = "Outputs/"+REGION+"/shape_v1_"+VERSION+model_mass
    f = rt.TFile(file_name+"_v1.root","UPDATE") 
    f.cd("signal_region")
#     h1.Write("zp_"+sys+"Up", rt.TObject.kWriteDelete)
#     h2.Write("zp_"+sys+"Down", rt.TObject.kWriteDelete)
    h1.Write()
    h2.Write()  

In [14]:
from collections import defaultdict
import math

c = rt.TCanvas()
c.SetGrid()
rt.gStyle.SetOptStat(0)
    
processes = ['TT', 'WW', 'zp']    
# def export_pdf_scale_unc(model_mass): 
for p in processes:
    for model_mass in model_masses:
        if model_mass!= 'Zp-2500_CH-345':
            continue
        if p=='zp':
            key = model_mass
            nset = 31
            score_key = 'score'+model_mass
            str_h = 'zp_pdf'
        else:
            key = p
            nset = 101
            score_key = 'score'+model_mass
            if p == 'TT':
                str_h = 'ttbar_pdf'
            if p == 'WW':
                str_h = 'ww_pdf'
        w0 = dic_predict_sort[key]['weight'].copy(deep=True)
        hists = []
        
        for j in range(nset):
            h = rt.TH1D("", "", nbin, 0, 1)
            for i, v in enumerate(dic_predict_sort[key][score_key]):
                w = w0[i]*dic_predict_sort[key]['pdf'+str(j)+'_pre'][i]
                h.Fill(v, w)
        #     h.Draw("same")
            hists.append(h)
        # print(hists)
    #     c.Draw()
    #     c.SaveAs("Syst/pdfs_"+model_mass+".pdf")

        dic_yields=defaultdict(list)
        for i, h in enumerate(hists):
            for nb in range(nbin):
                dic_yields["pdf"+str(i)].append(h.GetBinContent(nb+1))

        unc_pdfs = []
        for nb in range(nbin):
            detla_sqrs = []
            for i in range(nset-1):
                delta = dic_yields["pdf"+str(i+1)][nb] - dic_yields["pdf0"][nb]
                detla_sqr = delta*delta
                detla_sqrs.append(detla_sqr)
            sum_delta_sqrs = sum(detla_sqrs)
            unc_pdf = math.sqrt(sum_delta_sqrs)
            unc_pdfs.append(unc_pdf)

        hist=HistStack(dic_predict_sort, score_key, 'weight', nbin, 0, 1, "DNN score") 
        c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/normpdf"+model_mass)
        # print(hist.get_yields_by_bin()[model_mass])

        pdf_norm = np.array(hist.get_yields_by_bin()[key])
        # weight
        unc_pdfs1 = np.array(unc_pdfs)*pdf_norm[-1]/dic_yields["pdf0"][-1]

        print('process:', p, "mass:", model_mass, "norm:", pdf_norm[-1], dic_yields["pdf0"][-1], "new:", unc_pdfs1[-1], unc_pdfs[-1])

        pdf_up = pdf_norm + unc_pdfs1
        pdf_down = pdf_norm - unc_pdfs1

        h_pdfup = rt.TH1D(str_h+"Up", str_h+"Up", nbin, 0, 1)
        h_pdfdown = rt.TH1D(str_h+"Down", str_h+"Down", nbin, 0, 1)
        for nb in range(nbin):
            h_pdfup.SetBinContent(nb+1, pdf_up[nb])
            h_pdfdown.SetBinContent(nb+1, pdf_down[nb])
        
        if p == "zp":
            h_scaleUp = rt.TH1D("zp_scaleUp", "zp_scaleUp", nbin, 0, 1)
            h_scaleDown = rt.TH1D("zp_scaleDown", "zp_scaleDown", nbin, 0, 1)
            for i, v in enumerate(dic_predict_sort[model_mass]["score"+model_mass]):
                wup = w0[i]*dic_predict_sort[model_mass]['genweight_u_pre'][i]
                wdn = w0[i]*dic_predict_sort[model_mass]['genweight_d_pre'][i]
                h_scaleUp.Fill(v, wup)
                h_scaleDown.Fill(v, wdn)

            write_sys_hist(h_scaleUp, h_scaleDown)
        write_sys_hist(h_pdfup, h_pdfdown)



norm: 3.0572625800687456 3.057169279819109 new: 0.05145354057259063 0.051451970335146376
norm: 1.0327608241713049 1.032729306812169 new: 0.01932723903455366 0.01932664921402649


Info in <TCanvas::Print>: pdf file Outputs/SR/normpdfZp-2500_CH-345_scoreZp-2500_CH-345breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/normpdfZp-2500_CH-345_scoreZp-2500_CH-345.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/normpdfZp-2500_CH-345_sign_scoreZp-2500_CH-345.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file Outputs/SR/normpdfZp-4100_CH-345_scoreZp-4100_CH-345breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/normpdfZp-4100_CH-345_scoreZp-4100_CH-345.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/normpdfZp-4100_CH-345_sign_scoreZp-4100_CH-345.pdf has been created
